# Chapter 5: Pretraining on unlabeled data
This chapter covers the training and evaluation of a GPT model. It also looks at loading pretrained weights from OpenAI

### 5.1 Evaluating Generative Text Models
This chapter looks at how we can evaluate text generartion models

In [7]:
import tiktoken
import sys
sys.path.append("..")
from src.classes import *

In [7]:
# class MultiHeadAttention(nn.Module):
#     def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
#         super().__init__()
#         assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

#         self.d_out = d_out
#         self.num_heads = num_heads
#         self.head_dim = d_out // num_heads  # Reduce the projection dim to match desired output dim

#         self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
#         self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
#         self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
#         self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
#         self.dropout = nn.Dropout(dropout)
#         self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

#     def forward(self, x):
#         b, num_tokens, d_in = x.shape

#         keys = self.W_key(x)  # Shape: (b, num_tokens, d_out)
#         queries = self.W_query(x)
#         values = self.W_value(x)

#         # We implicitly split the matrix by adding a `num_heads` dimension
#         # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
#         keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
#         values = values.view(b, num_tokens, self.num_heads, self.head_dim)
#         queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

#         # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
#         keys = keys.transpose(1, 2)
#         queries = queries.transpose(1, 2)
#         values = values.transpose(1, 2)

#         # Compute scaled dot-product attention (aka self-attention) with a causal mask
#         attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

#         # Original mask truncated to the number of tokens and converted to boolean
#         mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

#         # Use the mask to fill attention scores
#         attn_scores.masked_fill_(mask_bool, -torch.inf)

#         attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
#         attn_weights = self.dropout(attn_weights)

#         # Shape: (b, num_tokens, num_heads, head_dim)
#         context_vec = (attn_weights @ values).transpose(1, 2)

#         # Combine heads, where self.d_out = self.num_heads * self.head_dim
#         context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
#         context_vec = self.out_proj(context_vec)  # optional projection

#         return context_vec


# class GELU(nn.Module):
#     def __init__(self):
#         super().__init__()

#     def forward(self, x):
#         return 0.5 * x * (1 + torch.tanh(
#             torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
#             (x + 0.044715 * torch.pow(x, 3))
#             ))


# class LayerNorm(nn.Module):
#     def __init__(self, emb_dim):
#         super().__init__()
#         self.eps    = 1e-5
#         self.scale  = nn.Parameter(torch.ones(emb_dim))
#         self.shift  = nn.Parameter(torch.zeros(emb_dim))

#     def forward(self, x):
#         mean    = x.mean(dim=-1, keepdim=True)
#         var     = x.var(dim=-1, keepdim=True)
#         norm_x  = (x - mean) / torch.sqrt(var + self.eps)
#         return self.scale * norm_x + self.shift 


# class FeedForward(nn.Module):
#     def __init__(self, cfg):
#         super().__init__()
#         self.layers = nn.Sequential(
#             nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
#             GELU(),
#             nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
#         )

#     def forward(self, x):
#         return self.layers(x)


# class TransformerBlock(nn.Module):

#     def __init__(self, cfg):
#         super().__init__()
#         self.att   = MultiHeadAttention(
#             d_in=cfg["emb_dim"],
#             d_out=cfg["emb_dim"],
#             context_length=cfg["context_length"],
#             num_heads=cfg["n_heads"],
#             dropout=cfg["drop_rate_multihead"],
#             qkv_bias=cfg["qkv_bias"]
#         )
#         self.ff    = FeedForward(cfg)
#         self.norm1 = LayerNorm(emb_dim=cfg["emb_dim"])
#         self.norm2 = LayerNorm(emb_dim=cfg["emb_dim"])
#         self.drop_shortcut = nn.Dropout(cfg["drop_rate_shortcut"])

#     def forward(self, x):
#         shortcut = x
#         x = self.norm1(x)
#         x = self.att(x)
#         x = self.drop_shortcut(x)
#         x = x + shortcut

#         shortcut = x
#         x = self.norm2(x)
#         x = self.ff(x)
#         x = self.drop_shortcut(x)
#         x = x + shortcut
#         return x
    

# class GPTModel(nn.Module):
#     def __init__(self, cfg):
#         super().__init__()
#         self.tok_emb    = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
#         self.pos_emb    = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
#         self.drop_emb   = nn.Dropout(cfg["drop_rate_embedding"])

#         self.trf_blocks = nn.Sequential(
#             *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
#             )
#         self.final_norm = LayerNorm(cfg["emb_dim"])
#         self.out_head   = nn.Linear(
#             cfg["emb_dim"], cfg["vocab_size"], bias=False
#         )

#     def forward(self, in_idx):
#         batch_size, seq_len = in_idx.shape
#         tok_embeds = self.tok_emb(in_idx)
#         pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))

#         x = tok_embeds + pos_embeds
#         x = self.drop_emb(x)
#         x = self.trf_blocks(x)
#         x = self.final_norm(x)

#         logits = self.out_head(x)
#         return logits


In [6]:
GPT_CONFIG_124M = {
    'vocab_size': 50257,
    'context_length': 256,
    'emb_dim': 768,
    'n_heads': 12,
    'n_layers': 12,
    'drop_rate_embedding': 0.1,
    'drop_rate_shortcut': 0.1,
    'drop_rate_multihead': 0.1,
    'qkv_bias': False,
    }

def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :] # Focus only on last timestep, so that (batch, n_token, vocab_size) -> (batch, vocab_size)
        probas = torch.softmax(logits, dim=-1) # Probabilities for token being a word in vocab
        idx_next = torch.argmax(probas, dim=-1, keepdim=True) # Find token with highest probability
        idx = torch.cat((idx, idx_next), dim=1) # Add token at last place in sequence for next prediction

    return idx

In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()

In [10]:
# Create functions for encoding and decoding
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # Adds batch dimension to tensor
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # Remove batch dimension
    return tokenizer.decode(flat.tolist())


In [13]:
start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
    )

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


#### 5.1.2 Calculating the Text Generation Loss
Since we haven't trained the model yet it clearly fails to output coherent text. For training it we need to measure the text generation loss, to see how the model behaves during training and if it's learning anything.

In [17]:
inputs = torch.tensor([[16833, 3626, 6100],  # ["every effort moves",
                       [40, 1107, 588]])     # "I really like"]

targets = torch.tensor([[3626, 6100, 345 ],  # [" effort moves you",
                        [1107, 588, 11311]]) # " really like chocolate"]

with torch.no_grad(): # Feed input into model to calculate logits
    logits = model(inputs)

probas = torch.softmax(logits, dim=-1) # Probability of each token in vocab
print(probas.shape)
token_ids = torch.argmax(probas, dim=-1, keepdim=True) # Get tokenID for higest scoring probability
print("Token IDs:\n", token_ids)

# Convert back into text
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1:"
 f" {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

torch.Size([2, 3, 50257])
Token IDs:
 tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])
Targets batch 1:  effort moves you
Outputs batch 1:  Armed heNetflix


In [19]:
# Initial softmax scores for target tokens
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 1:", target_probas_1)

text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 2:", target_probas_2)

Text 1: tensor([7.4514e-05, 3.1054e-05, 1.1567e-05])
Text 2: tensor([1.0343e-05, 5.6737e-05, 4.7620e-06])


In [24]:
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(log_probas)
avg_log_probas = torch.mean(log_probas)
print("Avergare loss score:", avg_log_probas)
neg_avg_log_probas = avg_log_probas * -1
print("Cross entropy loss", neg_avg_log_probas)

# Cross entropy measures the difference between two probability distributions, typically true distributions (tokens in dataset) vs predicted distribution.

tensor([ -9.5045, -10.3798, -11.3674, -11.4792,  -9.7771, -12.2549])
Avergare loss score: tensor(-10.7938)
Cross entropy loss tensor(10.7938)


In [27]:
print("Logits shape:", logits.shape)        # (batch, num_tokens, vocab_size)
print("Targets shape:", targets.shape)      # (batch, num_tokens)

Logits shape: torch.Size([2, 3, 50257])
Targets shape: torch.Size([2, 3])


In [26]:
# Since the logits and targets have different dimensions we need to flatten them over the batch dimension
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()
print("Flattened logits:", logits_flat.shape)
print("Flattened targets:", targets_flat.shape)

Flattened logits: torch.Size([6, 50257])
Flattened targets: torch.Size([6])


In [28]:
# Calculate cross entropy loss using Pytorch function
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

tensor(10.7938)
